## AReM Data
classify the activities of humans based on time series obtained by a Wireless Sensor Network

### Time Series Classification Part 1: Feature Creation/Extraction

### (a). Download the AReM data 

https://archive.ics.uci.edu/ml/datasets/label+Recognition+system+based+on+Multisensor+data+fusion+\%28AReM\%29. The dataset contains 7 folders that represent seven types of label. In each folder, there are multiple files each of which represents an instant of a human performing an label. Each file containis 6 time series collected from label of the same person, which are called avg_rss12, var_rss12, avg_rss13, var_rss13,vg_rss23, and ar_rss23. There are 88 instances in the dataset, each of which contains 6 time series and each time series has 480 consecutive values

In [ ]:
import os
import re
import csv
import numpy as np
import pandas as pd
import seaborn as sns
import math
import random
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')

In [ ]:
for path, dirs, files in os.walk('./data'):  # my data location
    print(path)
    print(dirs)

### (b). generate data sets

Keep datasets 1 and 2 in folders bending1 and bending 2, as well as datasets 1, 2, and 3 in other folders as test data and other datasets as train data.

In [ ]:
# create to list to save files
trainfiles = []
testfiles = []

sniffer = csv.Sniffer()

# please change the path to the github local path to avoid bug
# Here, path is current catalogue root; dirs is a list meaning mini_catalogue's name; file is a list meaning current all files' name.
for path, dirs, files in os.walk('./data'): 

    for file in files:
        if file.endswith(".csv"):
                file_path = os.path.join(path,file)
                if 'bending1' in file_path or 'bending2' in file_path:
                    if 'dataset1.csv' in file or 'dataset2.csv' in file:
                        testfiles.append(file_path)
                    else:
                        trainfiles.append(file_path)
                else:
                    if 'dataset1.csv' in file or 'dataset2.csv' in file or 'dataset3.csv' in file:
                        testfiles.append(file_path)
                    else:
                        trainfiles.append(file_path)
    
print(testfiles)
print(len(testfiles))
print(trainfiles)
print(len(trainfiles))

### (c). Feature Extraction. 
Classification of time series usually needs extracting features from them. In this problem, we focus on time-domain features.

(c) i. Research what types of time-domain features are usually used in time series classification and list them (examples are minimum, maximum, mean, etc).

Ans: 
- maximum
- minimum
- mean
- median
- standard deviation
- first quartile
- third quartile
- distribution
- correlation structure (but may not as important as the previous items)


(c) ii. Extract the time-domain features minimum, maximum, mean, median, standard deviation, first quartile, and third quartile for all of the 6 time series
in each instance. 

You are free to normalize/standardize features or use them directly. where, for example, 1st quart6, means the first quartile of the sixth time series in each of the 88 instances.

In [ ]:
orinal_name = ["# Columns: time","avg_rss12","var_rss12","avg_rss13","var_rss13","avg_rss23","var_rss23"]
col_names = ["time","avg_rss12","var_rss12","avg_rss13","var_rss13","avg_rss23","var_rss23"]
FEATURES = ['mean', 'std', 'min', '1st_quartile', 'median', '3rd_quartile', 'max']
feature_summary = ['min', 'max', 'mean', 'median', 'std', '1st_quartile', '3rd_quartile']

# take a look of our data
overview_df = pd.read_csv(testfiles[0], skiprows=5, header = None, on_bad_lines='skip')
overview_df.columns = col_names
overview_df

In [ ]:
file = './data/AReM/bending1/dataset1.csv'


define split function

In [ ]:
# prepare functions to split dataset and construc new data

def get_label(file):
    parent_directory = os.path.dirname(file)
    filename_components = parent_directory.split("/")
    return filename_components[-1]

# define the targeted columns
def targeted_column(segs):
    targeted_column = []
    for i in range(1, 6 * segs + 1):
        for j in feature_summary:
            targeted_column.append(f'{j}{i}')
    return targeted_column

def get_columns(segs):
    endnum = 6 * segs + 1
    columns = []
    for i in range(1,endnum):
        for stat in FEATURES:
            columns.append(f'{stat}{i}')
    return columns

    
def load_data(files, targeted_column, segs, standard=False):
    instances  = []
    labels = []
    for file in files:
        labels.append(get_label(file))
        
        # get the separator, since files have different separators
        with open(file, newline='') as source:
            separator = sniffer.sniff(source.read()).delimiter

        if separator == ",":
            origin_data = pd.read_csv(file, skiprows=5, header=None, on_bad_lines='skip')
        elif separator == "\\s+":
            origin_data = pd.read_csv(file, skiprows=5, sep="\\s+", header=None, on_bad_lines='skip')
        else:
            origin_data = pd.read_csv(file, skiprows=5, sep="\\s+", header=None, on_bad_lines='skip')
            
        origin_data.columns = orinal_name
        
        # Break the full time series into approximately equal segments.
        description = []
        for indices in np.array_split(np.arange(len(origin_data)), segs):
            cur_seg = origin_data.iloc[indices]
            time_col = "# Columns: time"
            cur_describe = cur_seg.describe().drop('count').drop(columns=time_col).T
            description.append(cur_describe.values.flatten())

        instances.append(np.concatenate(description))
            
    # reordering
    feature_df = pd.DataFrame(instances)
    feature_df.columns = get_columns(segs)
    feature_df = feature_df.loc[:, targeted_column]
    
    # label addition
    feature_df['label'] = pd.Series(labels)
    
    return feature_df

In [ ]:
targeted_column1 = targeted_column(segs=1)

train_data = load_data(trainfiles, targeted_column1, segs=1)
test_data = load_data(testfiles, targeted_column1, segs=1)

In [ ]:
train_data

In [ ]:
test_data

In [ ]:
final_data = pd.concat([train_data, test_data], axis=0)

(c) iii. Estimate the standard deviation of each of the time-domain features you extracted from the data. Then, use Python’s bootstrapped or any other method to build a 90% bootsrap confidence interval for the standard deviation of each feature.

In [ ]:
def bootstrap_std_confidence_interval(data, alpha=0.1, n_iterations=1000, seed=42):
    rng = np.random.default_rng(seed)
    values = data.to_numpy()
    bootstrap_std_values = [rng.choice(values, size=len(values), replace=True).std(ddof=1)
                            for _ in range(n_iterations)]
    
    # Calculate lower and upper percentiles directly
    lower = np.percentile(bootstrap_std_values, 100 * alpha / 2)
    upper = np.percentile(bootstrap_std_values, 100 * (1 - alpha / 2))
    
    return lower, upper

def process_std_interval(results):
    std_conf_intervals = {}
    for col in results.columns[:-1]:
        lower, upper = bootstrap_std_confidence_interval(results[col])
        std_conf_intervals[col] = {
            'std': results[col].std(),
            'lower_bound_90%_CI': lower,
            'upper_bound_90%_CI': upper,
        }

    # Convert the dictionary directly to a DataFrame
    std_conf_intervals_df = pd.DataFrame.from_dict(std_conf_intervals, orient='index')

    return std_conf_intervals_df


In [ ]:
std_conf_intervals_df = process_std_interval(final_data)
std_conf_intervals_df

In [ ]:
final_data.describe().loc['std']

(c) iv. Use your judgement to select the three most important time-domain features (one option may be min, mean, and max).

In [ ]:
sorted_std = final_data.describe().loc['std'].sort_values(ascending=False)
sorted_std

In [ ]:
#convert the label from the true value to 1/0 (bending/not bending )

df = pd.DataFrame(final_data)
df.loc[df['label'].isin(['bending1', 'bending2']), 'label'] = 1
df.loc[df['label'] != 1, 'label'] = 0

name = ['min', 'max', 'mean', 'median', 'std', '1st_quartile', '3rd_quartile']
num = ['1','2','3','4','5','6']

fig, axes = plt.subplots(7, 6, figsize=(30, 30))

for i in range(0,7):
    for j in range(0,6):
        bending = df.loc[df['label'] == 1, name[i]+num[j]]
        nobending = df.loc[df['label'] == 0, name[i]+num[j]]
        sns.distplot(bending, ax = axes[i, j], label='Bending', kde='True')
        sns.distplot(nobending, ax = axes[i, j], label='Not Bending', kde='True')
    
        
plt.show()

**Ans**: According to the picture showed above, as for this regression issue, driven by the considering of clear separated distribution. Also, we should consider denser distribution which means one class has some representitive values.

So the choice would be 3rd_quartile, mean and max. 

### 2. ISLR 3.7.4

I collect a set of data (n = 100 observations) containing a single predictor and a quantitative response. I then fit a linear regression model to the data, as well as a separate cubic regression, i.e. Y = β0 +β1X +β2X2 +β3X3 +ε.

**(a)** Suppose that the true relationship between X and Y is linear, i.e. Y = β0 + β1X + ε. Consider the training residual sum of squares (RSS) for the linear regression, and also the training RSS for the cubic regression. Would we expect one to be lower than the other, would we expect them to be the same, or is there not enough information to tell? Justify your answer.

**Ans:** Since predictors are directly proportional to the fitting of the model, the RSS for linear regression would be higher as compared to the RSS for cubic regression. Because cubic regression has more parameters to fit and more flexible, which might be better fit for training dataset.



**(b)** Answer (a) using test rather than training RSS.

**Ans:** More predictors generally leads to overfitting. But we also do not have enough information about the relationship between predictor and response. If their relationship is linear, for the test case, the RSS for linear regression might be lower since it is more likely to provide relatively correct regression result.



**(c)** Suppose that the true relationship between X and Y is not linear, but we don’t know how far it is from linear. Consider the training RSS for the linear regression, and also the training RSS for the cubic regression. Would we expect one to be lower than the other, would we expect them to be the same, or is there not enough information to tell? Justify your answer.

**Ans:** The fundamental principle does not change. More predictors lead to less RSS and hence, the RSS for cubic would be less. Cubic regression has more flexibility.


**(d)** Answer (c) using test rather than training RSS.

**Ans:** The information provided is insufficient as the answer will depend on the finding of which regression is the actual answer closer to.

# HW4-Time Series Classification Part 2: Binary and Multiclass Classification

## a) Binary Classification Using Logistic Regression

### i. use the training set to classify bending from other label 

Depict scatter plots of the features you specified in 1(c)iv extracted from time series 1, 2, and 6 of
each instance, and use color to distinguish bending vs. other label. (See p. 129 of the textbook)

Note: last HW3 I choose 3rd_quartile, mean and median, but later I reviewed and change the choice to 3rd_quartile, mean and max. Because max var seems have more clear separated distribution than median.

In [ ]:
def get_cols(indexes, props):
    cols = []
    for index in indexes:
        for prop in props:
            col_name = prop + str(index)
            cols.append(col_name)
    return cols

def get_labelled_cols(df, cols):
    temp = df['label']
    df.loc[temp.isin(['bending1', 'bending2']), 'label'] = 1
    df.loc[temp != 1, 'label'] = 0
    return cols + ['label']

def plot_scatter_plot(df, hue_value):
    sns.pairplot(df, hue=hue_value)
    # plt.legend(title='Bending - Non-bending Training data', loc='upper left', labels=['Non-Bending', 'Bending'])
    plt.show()

In [ ]:
# use training data here, take a look
train_data

In [ ]:
# extract 3rd_quartile, mean and max.
selected_col = get_cols([1,2,6], ['max', 'mean', '3rd_quartile'])
selected_labelled = get_labelled_cols(train_data, selected_col)
selected_train_df = train_data[selected_labelled]
selected_train_df

In [ ]:
plot_scatter_plot(selected_train_df, 'label')

### ii. Break each time series into two (approximately) equal length time series

Then repeat the 4(a)i experiment. Do you see any considerable difference in the results with those of 4(a)i?

In [ ]:
targeted_colomn2 = targeted_column(segs=2)

train_data_two = load_data(trainfiles, targeted_colomn2, segs=2)
test_data_two = load_data(testfiles, targeted_colomn2, segs=2)

In [ ]:
# note that 1-6 is the previous half time series, 7-12 is the last half time series

selected_col = get_cols([1, 2, 6, 7, 8, 12], ['max', 'mean', '3rd_quartile'])
selected_labelled = get_labelled_cols(train_data_two, selected_col)
selected_train_df_2nd = train_data_two[selected_labelled]
selected_train_df_2nd

In [ ]:
plot_scatter_plot(selected_train_df_2nd[['max1', 'max7','max2', 'max8','max6', 'max12',
                                         'mean1', 'mean7','mean2', 'mean8','mean6', 'mean12',
                                         '3rd_quartile1', '3rd_quartile7','3rd_quartile2', '3rd_quartile8', '3rd_quartile6', '3rd_quartile12',
                                         'label']], 'label')

**Ans** 

It seems no explicit difference after the split. 

### iii. Break each time series in your training set into l ∈ {1,2,...,20} time series
of approximately equal length and use logistic regression to solve the binary classification problem, using time-domain features

Remember that breaking
each of the time series does not change the number of instances. It only
changes the number of features for each instance. Calculate the p-values for
your logistic regression parameters in each model corresponding to each value
of l and refit a logistic regression model using your pruned set of features.6
Alternatively, you can use backward selection using sklearn.feature selection
or glm in R. Use 5-fold cross-validation to determine the best value of the pair
(l,p), where p is the number of features used in recursive feature elimination.
Explain what the right way and the wrong way are to perform cross-validation

In [ ]:
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_selection import RFECV
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import confusion_matrix, roc_curve, auc, accuracy_score
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB

In [ ]:
# set an additional function for oversample, set default state as "False"
def oversample_process(data, random_state=60):
    print('Execute the oversample_process')
    # Count the instances of each class
    activity_counts = data['label'].value_counts()
    
    # Identify the minority and majority classes
    min_activity = activity_counts.idxmin()
    max_activity = activity_counts.idxmax()
    
    # Perform resampling
    min_ds = resample(data[data['label'] == min_activity],
                      replace=True,
                      n_samples=activity_counts[max_activity],
                      random_state=random_state)
    
    # Combine the resampled minority class with the majority class data
    data_balanced = pd.concat([data[data['label'] == max_activity], min_ds])
    
    return data_balanced


# split feature and labeling 
def getxy(data):
    x = data.iloc[:, :-1]
    y = data.iloc[:, -1]
    return x, y


def binary_labels(labels):
    return np.where(labels.isin(['bending1', 'bending2']), 1, 0)


def logistic_regression(trainfiles, l, fold_count, penalty=None):
    print(f"Calculate when l = {str(l)}:")
    
    new_column = targeted_column(l) # l is segs
    new_df = load_data(trainfiles, new_column, segs=l)
    
    new_df['label'] = binary_labels(new_df['label'])
    
    trainx, trainy = getxy(new_df)
    solver = 'liblinear' if penalty == 'l1' else 'lbfgs'
    
    # generate the subset for validation
    validator = StratifiedKFold(n_splits=fold_count, shuffle=True, random_state=69)
    
    # the regression model
    model = LogisticRegression(penalty=penalty, solver=solver, max_iter=100)
    
    # model selection, combine with cross-validation, return the selecter feature
    selector = RFECV(estimator=model, cv=validator, scoring='accuracy')
    
    selector.fit(trainx, trainy)
    
    return model, selector


In [ ]:
fold_n = 5
L = 20

# usa new dic to save cross validation result
combination = {'number of l': [],'number of feature': [], 'score': []}

# use greedy search here, did not use backward parameter
for l in range(1, L+1):
    model, selector = logistic_regression(trainfiles, l, fold_n)
    num_f = selector.n_features_
    score = selector.cv_results_['mean_test_score'].max()
    combination['number of l'].append(l)
    combination['number of feature'].append(num_f)
    combination['score'].append(round(score, 4))

df = pd.DataFrame(combination)
best_binary_l = int(df.loc[df['score'].idxmax(), 'number of l'])
df

In [ ]:
# Since the value of l corresponding to the highest score is 1, I choose l=1 to train the model and calculate the ROC and AUC.

def selected_features(columns, selector):
    selected_features = []
    for i, feature in enumerate(columns):
        if selector.support_[i]:
            selected_features.append(feature)
    return selected_features

model, opti_selector = logistic_regression(trainfiles, best_binary_l, 5)
opti_features = selected_features(targeted_column(segs=best_binary_l), opti_selector)

In [ ]:
# show the selected feature
opti_features

### iv. Report the confusion matrix and show the ROC and AUC

for your classifier on train data. Report the parameters of your logistic regression betas as well as the p-values associated with them.

In [ ]:
train_selected = load_data(trainfiles, opti_features, segs=best_binary_l)
train_selected['label'] = binary_labels(train_selected['label'])
train_selected

In [ ]:
# based on the selected_value to train the model again and show the ROC picture 
trainx_selected, trainy_selected = getxy(train_selected)

# ROC and AUC
def roc_plot(ytrue, ypred_prob, classifier):
    fig, axes = plt.subplots(figsize=(10, 8))
    axes.set_title("ROC")
    for i, j in enumerate(classifier.classes_):
        fpr, tpr, thresholds = roc_curve(ytrue, ypred_prob[:, i], pos_label=j)
        roc_auc = auc(fpr, tpr)
        axes.plot(fpr, tpr, label=f"Class {j}: AUC {roc_auc:.4f}")
    axes.plot([0, 1], [0, 1], "--")
    axes.set_xlabel("False Positive Rate")
    axes.set_ylabel("True Positive Rate")
    axes.legend()
    plt.show()

def evaluate_fitted_classifier(x, y, features, classifier):
    ypred_prob = classifier.predict_proba(x[features])
    ytrue = y.to_numpy(dtype=float)
    ypred = classifier.predict(x[features]).astype(int)
    roc_plot(ytrue, ypred_prob, classifier)
    
    # confusion matrix
    cm = confusion_matrix(ytrue, ypred)
    print(f"Confusion Matrix:\n{cm}")
    
    # accuracy
    print(f'accuracy score = {accuracy_score(ytrue, ypred)}')


# Fit the selected model once on training data, then reuse it unchanged for test evaluation.
selected_model = LogisticRegression(max_iter=100, random_state=69)
selected_model.fit(trainx_selected[opti_features], trainy_selected)
evaluate_fitted_classifier(trainx_selected, trainy_selected, opti_features, selected_model)

In [ ]:
df_trainx_selected = trainx_selected.loc[:, opti_features]
id_variable = sm.add_constant(df_trainx_selected.to_numpy(dtype=float))
logiticmodel_l1 = sm.Logit(trainy_selected.to_numpy(dtype=float), id_variable)
summary = logiticmodel_l1.fit(method='bfgs').summary()

print(summary)

### v. Test the classifier on the test set. 

Remember to break the time series in your test set into the same number of time series into which you broke your training set. Remember that the classifier has to be tested using the features extracted from the test set. Compare the accuracy on the test set with the cross-validation accuracy you obtained previously.

In [ ]:
test_selected = load_data(testfiles, opti_features, segs=best_binary_l)
test_selected['label'] = binary_labels(test_selected['label'])
testx_selected, testy_selected = getxy(test_selected)

evaluate_fitted_classifier(testx_selected, testy_selected, opti_features, selected_model)

**Ans**

the accuracy in both test and cross-valication are almost identical

### vi. Do your classes seem to be well-separated to cause instability in calculating logistic regression parameters?

**Ans:** 
- Yes. All the p-value in each predictors are unsignificant, suggesting that there is **Complete Separation**. In this case the Maximum Likelihood Estimator does not exist and the parameters are not identified.


- Due to this finding, it is possible that the well-separation of the classes is cauisng the instability in calculation o fthe regression parameters.

### vii) From the confusion matrices you obtained, do you see imbalanced classes? 

If yes, build a logistic regression model based on case-control sampling and adjust its parameters. Report the confusion matrix, ROC, and AUC of the model.

**Ans:**

- Yes. 0 = non-bending classes and 1 = bending classes. The confusion matrices shows that there are 69 instances of non-bending classes and 9 instances for bending classes, which means the classes are definitely imbalanced.

- The following step I'll build model based on case-control and adjust parameters

**vii) case control sampling logistic regression**

In [ ]:
# ponytail: reuse training-only CV features; add fold-local resampling if oversampled model selection is needed.
# Oversampling is applied only when fitting the final case-control model.
classifier = LogisticRegression(max_iter=100, random_state=69)
best_binary_l, opti_features

In [ ]:
opti_features

In [ ]:
training_selected = load_data(trainfiles, opti_features, segs=best_binary_l)
training_selected['label'] = binary_labels(training_selected['label'])

train_set = oversample_process(training_selected)
trainX, trainY = getxy(train_set)

classifier.fit(trainX[opti_features], trainY.to_numpy(dtype=int))
evaluate_fitted_classifier(trainX, trainY, opti_features, classifier)

In [ ]:
testing_selected = load_data(testfiles, opti_features, segs=best_binary_l)
testing_selected['label'] = binary_labels(testing_selected['label'])

# ATT: here, for test dataset, should not execute oversample process

testX, testY = getxy(testing_selected)

evaluate_fitted_classifier(testX, testY, opti_features, classifier)

## (b) Binary Classification Using L1-penalized logistic regression

### i. Repeat 2(a)iii using L1-penalized logistic regression

i.e. instead of using p- values for variable selection, use L1 regularization. Note that in this problem, you have to cross-validate for both l, the number of time series into which you break each of your instances, and λ, the weight of L1 penalty in your logistic regression objective function (or C, the budget). Packages usually perform cross-validation for λ automatically.9

In [ ]:
def classifyirr(classifier_obj, kwargs, binary_classes):
    results = []
    for l in range(1, 20 + 1):
        
        print("When l = " + str(l))
        new_column = targeted_column(l)
        
        new_traindata = load_data(trainfiles, new_column, segs=l)
        new_traindata = pd.DataFrame(new_traindata)
        
        if binary_classes:
            new_traindata['label'] = binary_labels(new_traindata['label'])
        
        new_trainx, new_trainy = getxy(new_traindata)
        
        # Select l using training data only. The held-out test set is evaluated once in usebestopt().
        classifier = classifier_obj(**kwargs)
        validator = StratifiedKFold(n_splits=5, shuffle=True, random_state=69)
        cv_score = cross_val_score(classifier, new_trainx, new_trainy,
                                   cv=validator, scoring='accuracy').mean()
        results.append((l, cv_score))
        print("Results when l = {} : mean CV score {}".format(l, round(cv_score, 4)))
    return pd.DataFrame(results, columns=['l', 'cv_score'])
 

# parameter settings to create model
kwargs = {
    'penalty' : 'l1', 
    'max_iter' : 100,
    'scoring' : 'accuracy',
    'cv' : 5,
    'random_state' : 69,
    'solver' : 'liblinear'
}

binary_cv = classifyirr(LogisticRegressionCV, kwargs, binary_classes=True)
best_binary_l = int(binary_cv.loc[binary_cv['cv_score'].idxmax(), 'l'])

In [ ]:
def roc_plot(ytrue, ypred_prob, classifier):
    fig, axes = plt.subplots(figsize=(10, 8))
    axes.set_title("ROC")
    for i, j in enumerate(classifier.classes_):
        fpr, tpr, thresholds = roc_curve(ytrue, ypred_prob[:, i], pos_label=j)
        roc_auc = auc(fpr, tpr)
        axes.plot(fpr, tpr, label=f"Class {j}: AUC {roc_auc:.4f}")
    axes.plot([0, 1], [0, 1], "--")
    axes.set_xlabel("False Positive Rate")
    axes.set_ylabel("True Positive Rate")
    axes.legend()
    plt.show()

def usebestopt(l, classifier_obj, kwargs, binary_classes):

    new_column = targeted_column(l)
    new_traindata = load_data(trainfiles, new_column, segs=l) 
    new_testdata = load_data(testfiles, new_column, segs=l)
    
    if binary_classes:
        new_traindata['label'] = binary_labels(new_traindata['label'])
        new_testdata['label'] = binary_labels(new_testdata['label'])
        
    new_trainx, new_trainy = getxy(new_traindata)
    new_testx, new_testy = getxy(new_testdata)
        
    # build classifier to fit
    classifier = classifier_obj(**kwargs)
    classifier.fit(new_trainx, new_trainy)
        
    # get prediction probability for both train and test, this is used to get ROC curve
    train_predictY_prob = classifier.predict_proba(new_trainx)
    predictY_prob = classifier.predict_proba(new_testx)
    
    # get prediction of test set and its confusion matrix
    predictY = classifier.predict(new_testx)
    conf_mat = confusion_matrix(new_testy, predictY)
   
    
    print("trainset ROC Plot")
    roc_plot(new_trainy, train_predictY_prob, classifier)
    
    print("testset ROC Plot")
    roc_plot(new_testy, predictY_prob, classifier)
    
    print("Confusion matrix on test set:\n{}".format(conf_mat))
    
    
    
# Evaluate the held-out test set once, after selecting l by training-only CV.
kwargs = {
    'penalty' : 'l1', 
    'max_iter' : 100, 
    'solver' : 'liblinear'
}

usebestopt(best_binary_l, LogisticRegression, kwargs, binary_classes=True)

### ii. Compare the L1-penalized with variable selection using p-values. 

Which one performs better? Which one is easier to implement?

**Ans**

The L1-penalized logistic regression has better performance in terms of accuracy on the test data (approximately ~ 1.0). So the L1-penalized performs better. The L1-regularization is easier to implement as feature selection does not need to be manual.

## (c) Multi-class Classification (The Realistic Case)

i. Find the best l in the same way as you found it in 2(b)i to build an L1-penalized multinomial regression model to classify all activities in your training set cross-validation for λ automatically.

In [ ]:
kwargs = {
    'penalty' : 'l1', 
    'max_iter' : 100,
    'multi_class' : 'multinomial',
    'cv' : 5,
    'random_state' : 69,
    'solver' : 'saga'
}

multinomial_cv = classifyirr(LogisticRegressionCV, kwargs, binary_classes=False)
best_multinomial_l = int(multinomial_cv.loc[multinomial_cv['cv_score'].idxmax(), 'l'])

Select **l** by the highest training cross-validation score above, then evaluate the held-out test set once.

In [ ]:
kwargs = {
    'penalty' : 'l1', 
    'max_iter' : 100, 
    'multi_class' : 'multinomial', 
    'solver' : 'saga'
}

In [ ]:
usebestopt(best_multinomial_l, LogisticRegression, kwargs, binary_classes=False)

ii) Repeat 2(c)i using a Naive Bayes' classifier. Use both Gaussian and Multinomial priors and compare the results.

In [ ]:
gaussian_cv = classifyirr(GaussianNB, {}, binary_classes=False)
best_gaussian_l = int(gaussian_cv.loc[gaussian_cv['cv_score'].idxmax(), 'l'])

For the Gaussian prior, select **l** using the highest training cross-validation score.

In [ ]:
usebestopt(best_gaussian_l, GaussianNB, {}, binary_classes=False)

In [ ]:
# for MultinomiaNB
multinomial_nb_cv = classifyirr(MultinomialNB, {}, binary_classes=False)
best_multinomial_nb_l = int(multinomial_nb_cv.loc[multinomial_nb_cv['cv_score'].idxmax(), 'l'])

For the Multinomial prior, select **l** using the highest training cross-validation score.

In [ ]:
usebestopt(best_multinomial_nb_l, MultinomialNB, {}, binary_classes=False)

iii. Which method is better for multiclass classification in this problem?

**Ans**

The three models have the same error rate on test set. Nevertheless, the logistic regression displays superior ROC curves, with higher AUC values across all categories.

# 3) ISLR 4.8.3

![image.png](Q4-8-3.jpg)

# 3) ISLR 4.8.7

![image.png](Q4-8-7.jpg)